# MNIST 3 vs 7 from Scratch
This notebook rebuilds the classic MNIST 3 vs 7 classification without fast.ai helpers. Each section follows the required outline: download and normalize the dataset manually, construct the tensors, define a linear model with custom loss, implement a training loop that mimics DataLoader behavior, inspect gradients explicitly, and finally report validation metrics plus plots.

## 1. Data preparation without fast.ai
Download the MNIST_SAMPLE archive from fast.ai, explore the directory structure under `train/` and `valid/`, and manually convert each PNG to a normalized tensor using PIL and NumPy. This lays the foundation before introducing any model logic.

In [ ]:
import urllib.request
import tarfile
from pathlib import Path
from PIL import Image
import numpy as np
import torch

data_root = Path("/Users/manojkumar/workspace/project26/data")
data_root.mkdir(parents=True, exist_ok=True)
archive_url = "http://files.fast.ai/data/examples/mnist_sample.tgz"
archive_path = data_root / "mnist_sample.tgz"

if not archive_path.exists():
    print("Downloading MNIST_SAMPLE archive...")
    urllib.request.urlretrieve(archive_url, archive_path)

extract_path = data_root / "mnist_sample"
if not extract_path.exists():
    print("Extracting MNIST_SAMPLE...")
    with tarfile.open(archive_path) as archive:
        archive.extractall(data_root)

train_path = extract_path / "train"
valid_path = extract_path / "valid"
print("Train labels:", sorted(p.name for p in train_path.iterdir() if p.is_dir()))
print("Valid labels:", sorted(p.name for p in valid_path.iterdir() if p.is_dir()))

def load_digit_tensors(split_path, digit):
    digit_path = split_path / str(digit)
    tensors = []
    for file in sorted(digit_path.glob("*.png")):
        img = Image.open(file).convert("L")
        arr = torch.tensor(np.array(img), dtype=torch.float32) / 255.0
        tensors.append(arr)
    print(f"Loaded {len(tensors)} examples for digit {digit} from {split_path.name}")
    return tensors

train_3 = load_digit_tensors(train_path, 3)
train_7 = load_digit_tensors(train_path, 7)
valid_3 = load_digit_tensors(valid_path, 3)
valid_7 = load_digit_tensors(valid_path, 7)

## 2. Tensor operations for MNIST 3 vs 7
Stack the digit tensors to create rank-3 data structures, flatten them to rank-2 matrices for model input, and build label tensors. While doing so, explain why rank and broadcasting matter, and turn the tensors into Python Dataset-like tuples.

In [ ]:
train_tensor_3 = torch.stack(train_3)
train_tensor_7 = torch.stack(train_7)
valid_tensor_3 = torch.stack(valid_3)
valid_tensor_7 = torch.stack(valid_7)

train_x = torch.cat([train_tensor_3, train_tensor_7], dim=0).view(-1, 28 * 28)
train_y = torch.cat([torch.ones(len(train_tensor_3)), torch.zeros(len(train_tensor_7))]).unsqueeze(1)
valid_x = torch.cat([valid_tensor_3, valid_tensor_7], dim=0).view(-1, 28 * 28)
valid_y = torch.cat([torch.ones(len(valid_tensor_3)), torch.zeros(len(valid_tensor_7))]).unsqueeze(1)

print("train_x rank:", train_x.ndim, "shape:", train_x.shape)
print("train_y rank:", train_y.ndim, "shape:", train_y.shape)
print("Broadcast example (train_x[0] + 0.1):", (train_x[0] + 0.1).shape)

from torch.utils.data import DataLoader

train_dataset = list(zip(train_x, train_y))
valid_dataset = list(zip(valid_x, valid_y))
train_dl = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_dl = DataLoader(valid_dataset, batch_size=256)
print("Created DataLoader-like iterators for manual batching.")

## 3. Manual linear model, sigmoid, and loss
Initialize weight and bias tensors with `requires_grad_`, implement a linear prediction with matrix multiplication, wrap it with a sigmoid, and define a custom MNIST loss using `torch.where` to ensure the loss behaves smoothly even when predictions are not bounded yet.

In [ ]:
def init_params(shape, std=1.0):
    return torch.randn(shape, requires_grad=True) * std

weights = init_params((28 * 28, 1))
bias = init_params(1)

def linear_model(xb):
    return xb @ weights + bias

def mnist_loss(preds, targets):
    preds = preds.sigmoid()
    return torch.where(targets == 1, 1 - preds, preds).mean()

with torch.no_grad():
    sample_loss = mnist_loss(linear_model(train_x[:8]), train_y[:8])
print("Initial sample loss (before training):", sample_loss.item())

## 4. Custom mini-batch training loop
Implement a custom training loop that mimics an optimizer: compute predictions, evaluate the loss, call `backward()`, update the parameters using their gradients and the learning rate, and make sure gradients are zeroed after every update. Track the training loss and validation accuracy history for later inspection.

In [ ]:
params = [weights, bias]

def evaluate_accuracy(dl):
    accs = []
    for xb, yb in dl:
        preds = linear_model(xb).sigmoid()
        correct = (preds > 0.5) == yb
        accs.append(correct.float().mean().item())
    return sum(accs) / len(accs)

def train_epoch(dl, lr):
    losses = []
    for xb, yb in dl:
        preds = linear_model(xb)
        loss = mnist_loss(preds, yb)
        loss.backward()
        with torch.no_grad():
            for p in params:
                p.data -= lr * p.grad
        for p in params:
            p.grad.zero_()
        losses.append(loss.item())
    return sum(losses) / len(losses)

lr = 0.1
epochs = 5
loss_history = []
accuracy_history = []

for epoch in range(epochs):
    epoch_loss = train_epoch(train_dl, lr)
    loss_history.append(epoch_loss)
    val_acc = evaluate_accuracy(valid_dl)
    accuracy_history.append(val_acc)
    print(f"Epoch {epoch + 1}/{epochs}: loss={epoch_loss:.4f}, valid_acc={val_acc:.4f}")

## 5. Gradient inspection during training
Demonstrate how gradients accumulate when `backward()` is called multiple times on the same batch, how they look before and after zeroing, and why zeroing prevents gradient graphs from growing uncontrollably.

In [ ]:
diag_w = init_params((28 * 28, 1))
diag_b = init_params(1)

def diag_model(x): return x @ diag_w + diag_b

xb, yb = next(iter(train_dl))
loss1 = mnist_loss(diag_model(xb), yb)
loss1.backward()
print("Gradient norm after first backward:", diag_w.grad.norm().item())

loss2 = mnist_loss(diag_model(xb), yb)
loss2.backward()
print("Gradient norm after second backward (without zero):", diag_w.grad.norm().item())

diag_w.grad.zero_()
diag_b.grad.zero_()
print("Gradient norm after zeroing:", diag_w.grad.norm().item())

## 6. Validation accuracy and metrics
Use the sigmoid outputs to threshold predictions, compute validation accuracy, and plot the loss/accuracy progression across epochs to visually track how training evolves.

In [ ]:
import matplotlib.pyplot as plt

final_val_acc = evaluate_accuracy(valid_dl)
print(f"Final validation accuracy (recomputed): {final_val_acc:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(range(1, epochs + 1), loss_history, marker="o", label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Progression")
plt.grid(True)
plt.legend()

plt.figure(figsize=(6, 4))
plt.plot(range(1, epochs + 1), accuracy_history, marker="o", color="tab:orange", label="Valid Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Progression")
plt.grid(True)
plt.legend()
plt.show()